# Analyze $c_L$ and frequency spectrum for URANS with pitching

In [ ]:
import matplotlib.pyplot as plt

from os import makedirs
from os.path import join, exists
from torch import where, tensor, argmax, from_numpy, pi, sin, deg2rad

from utils import load_force_coeffs, interpolate_uniform, compute_fft

In [ ]:
# validation @ Ma = 0.73
u_inf = 242.16629
alpha0 = 3.5

# chord length
chord = 1

# Max. amplitude at f = 14.42473 Hz (Sr = 0.06175) for no pitching case (avg. over 218 CTU)
f_b = 14.42473
t_pitching_start = 0.06

# start of quasi-steady state -> chosen based on course of cl
tstar_start = 145

# use latex fonts
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360, "text.latex.preamble": r"\usepackage{xcolor}"})

# use default color cycle
color = [f"C{i}" for i in range(10)]

In [ ]:
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation")
save_dir = join("..", "run", "plots", "URANS_pitching", "URANS_blockMesh", "first_test_3.5deg")
cases = ["URANS_2D_Ma0.73_Re3e6/URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod_newMesh",
         "URANS_2D_Ma0.73_Re3e6_pitching/A0.875_f10", "URANS_2D_Ma0.73_Re3e6_pitching/A0.875_f14"]

# legend entries, first one is without pitching as comparison
legend = [r"$\mathrm{no~pitching}$"]

# compute delta alpha / alpha0  and f_pitching / f_buffet
amps, freqs = [], []
for i, c in enumerate(cases[1:]):
    amps.append(float(c.split("/")[-1].split("_")[0].strip("A")))
    freqs.append(float(c.split("/")[-1].split("_")[1].strip("f")))

    _fmt = "{:.2f}".format(amps[i] / alpha0), "{:.3f}".format(freqs[i] / f_b)
    legend += [rf"$\Delta \alpha \, / \, \alpha_0 = {_fmt[0]}, f_p \, / \, f_b = {_fmt[1]}$"]

In [ ]:
# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
def compute_aoa(time_steps, aoa0, amplitude, frequency, phase = 0):
    # verified with actual inflow angle from the simulations (in ParaView)
    return aoa0 + amplitude * sin(2*pi*frequency*time_steps - deg2rad(tensor(phase)))

In [ ]:
# load cl and cd
forces = [load_force_coeffs(join(load_dir, c)) for c in cases]

In [ ]:
# plot only cl
fig, ax = plt.subplots(nrows=2, figsize=(6, 4), sharex="col", gridspec_kw={"height_ratios": [2, 1]})
for i in range(len(cases)):
    ax[0].plot(forces[i].t * u_inf / chord, forces[i].cy, label=legend[i], color=color[i], zorder=10)
    # ax[0].plot(forces[i].t, forces[i].cy, label=legend[i], color=color[i])

    # compute the instantaneous AoA, omit case without pitching
    if i == 0:
        continue
    else:
        idx_start = where(abs(tensor(forces[i].t.values) - t_pitching_start) < 1e-8)[0][0].item()

        # note that there is a time delay of ~25 CTU until the inlet signal (plotted here) actually affects the cl
        aoa = compute_aoa(from_numpy(forces[i].t.values), alpha0, amps[i-1], freqs[i-1])

        # ax[1].plot(forces[i].t[idx_start:] * u_inf / chord, (aoa[idx_start:] - alpha0) / alpha0, color=color[i])
        ax[1].plot(forces[i].t[idx_start:] * u_inf / chord, aoa[idx_start:] , color=color[i])

ax[0].set_ylabel(r"$c_l$")
# ax[1].set_ylabel(r"$\Delta \alpha_\mathrm{inlet} \, / \, \alpha_0$")
ax[1].set_ylabel(r"$\alpha_\mathrm{inlet} \quad [^\circ]$")
ax[-1].set_xlabel(r"$t \frac{U_{\infty}}{c}$")
ax[-1].set_xlim(0, forces[-1].t.iloc[-1] * u_inf / chord)
ax[0].set_ylim(0.7, 1.04)

fig.tight_layout()
for i in range(2):
    ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="y")
    ax[i].minorticks_on()
    ax[i].tick_params(axis="x", which="minor", bottom=False)
    ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="y")

ax[0].axvline(t_pitching_start * u_inf /chord, ls=":", color="red", label=r"$\mathrm{pitching~start}$", zorder=10)
ax[0].fill_between([0, tstar_start], 0, 2, alpha=0.3, color="black", edgecolor="none", label=r"$\mathrm{discarded}$")

fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, "comparison_cl_vs_t.png"))
plt.show()

In [ ]:
# re-sample the force coefficients since the simulation is run with a variable time step
t_uniform, cy_inter = [], []
for f in forces:
    try:
        idx_start = where(abs(tensor(f["t"].values) * u_inf/chord - tstar_start) < 1e-4)[0][0].item()

        # remove the transient phase and duplicates (resulting from dt < write precision)
        f.drop(list(range(0, idx_start)), inplace=True)
        f.reset_index(inplace=True, drop=True)
    except IndexError:
        print("fail")
        
    f.reset_index(inplace=True, drop=True)

    # interpolate values
    _t_uniform, _cy_inter = interpolate_uniform(f["t"].values, f["cy"].values)
    t_uniform.append(_t_uniform)
    cy_inter.append(_cy_inter)

In [ ]:
# compute FFT
results_fft_cl = [compute_fft(cy_inter[c], (t_uniform[c][1] - t_uniform[c][0]).item()) for c in range(len(cases))]

idx = [argmax(from_numpy(res[1])) for res in results_fft_cl]

# print out buffet frequencies for each case
for i, res in enumerate(results_fft_cl):
    print(f"Max. amplitude at f = {round(res[0][idx[i]].item(), 5)} Hz (Sr = {round(res[0][idx[i]].item() * chord / u_inf, 5)})")

In [ ]:
# plot frequency of max. amplitude as Sr and Hz
f_legend, sr_legend = [], []

fig, ax = plt.subplots(2, 1, figsize=(6, 5), sharey="col")
for i, fa in enumerate(results_fft_cl):
    ax[0].plot(fa[0] * 2 * pi * chord/u_inf, fa[1], label=legend[i])
    ax[1].plot(fa[0] * chord / u_inf, fa[1])

    # add pitching frequency
    if i > 0:
        ax[0].axvline(freqs[i-1] * 2 * pi * chord/u_inf, color=color[i], ls=":", zorder=10)
        ax[1].axvline(freqs[i-1] * chord / u_inf, color=color[i], ls=":", zorder=10)

    # not working -> remains black
    f_legend.append(r"$\textcolor{" + color[i] + "}{f = " + "{:.2f}".format(fa[0][idx[i]].item()) + r"\, Hz" + "}$")
    sr_legend.append(r"$\textcolor{" + color[i] + "}{Sr = " + "{:.4f}".format(fa[0][idx[i]].item() * chord / u_inf) + "}$")

# ax[0].set_xscale("log")
ax[0].set_xlim(0, 1)
ax[1].set_xlim(0.02, 0.14)
ax[0].set_xlabel(r"$\frac{2\pi f c}{U_\infty}$ $[-]$")
ax[1].set_xlabel("$Sr$ $[-]$")

for i in range(2):
    ax[i].set_ylabel(r"$\mathrm{PSD}$")
    ax[i].ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

# mark Sr of literature
ax[0].axvspan(0.056 * 2 * pi, 0.076 * 2 * pi, 0, 1, color="black", alpha=0.3, label=r"$\mathrm{Kleinert~et~al.}$", zorder=0)
ax[1].axvspan(0.056, 0.076, 0, 1, color="black", alpha=0.3, zorder=0)

fig.tight_layout()
fig.legend(ncols=2, loc="upper center")
fig.subplots_adjust(top=0.84)
plt.savefig(join(save_dir, "dominant_frequency.png"))
plt.show()

In [ ]:
# plot complete FFT
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for i, res in enumerate(results_fft_cl):
    ax.plot(res[0] * chord / u_inf, res[1], label=legend[i], color=color[i])

    # add pitching frequency
    if i > 0:
        ax.axvline(freqs[i-1] * chord / u_inf, color=color[i], ls=":", zorder=10)

ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel(r"$Sr$ $[\mathrm{-}]$")
ax.set_ylabel(r"$\mathrm{PSD}$")
ax.set_ylim(1e-10, 1e-2)
ax.set_xlim(1e-2, 1e1)
fig.tight_layout()
ax.grid(visible=True, which="major", linestyle="-", alpha=0.45, color="black", axis="x")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.35, color="black", axis="x")
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, "results_fft.png"))
plt.show()